# Experiment 1 — Authority direction (§4)

Parametrized notebook: set MODEL_KEY in the config cell, then Run All on an A100.

For Exp 2-4, make sure Experiment 1 has already been run for the chosen model (the result bundle is read from Drive).


In [1]:
# Clone repo (for data/ + tests/) and install it in editable mode
import os, sys, importlib, site
REPO_DIR = "/content/Mech_spoof"

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/ChuloIva/Mech_spoof.git {REPO_DIR}

!pip install -q -e {REPO_DIR}

# Make sure mech_spoof resolves data/ to the cloned repo
os.environ["MECH_SPOOF_ROOT"] = REPO_DIR

# Ensure the current kernel picks up the freshly installed package
# (Colab caches sys.path before pip runs; editable installs need a nudge)
src_path = f"{REPO_DIR}/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
site.main()

import mech_spoof
print("mech_spoof loaded from:", mech_spoof.__file__)

In [3]:
# Auth + Drive
import os, getpass
from google.colab import drive, userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
try:
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
except Exception:
    pass
if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ').strip()
drive.mount('/content/drive')

In [6]:
!pip install --upgrade transformers

In [ ]:
import os, getpass                                                                                                                                                                       
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter key: ').strip()                                                                                                         
print('len:', len(os.environ['OPENROUTER_API_KEY']))  # non-zero confirms set 

In [1]:
# Config (EDIT ME)
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS

MODEL_KEY = 'qwen'            # qwen | llama3 | mistral | gemma | phi3
EXPERIMENT = 2
DRIVE_ROOT = Path('/content/drive/MyDrive/mech_spoof_results')

slug = MODEL_CONFIGS[MODEL_KEY].slug
exp_dirname = {1: 'exp1_authority', 2: 'exp2_conflict', 3: 'exp3_refusal', 4: 'exp4_attacks'}[EXPERIMENT]
OUT_DIR = DRIVE_ROOT / slug / exp_dirname
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR =', OUT_DIR)

In [2]:
# Run the experiment
if EXPERIMENT == 1:
    from mech_spoof.experiments.exp1_authority import run_experiment_1
    result = run_experiment_1(MODEL_KEY, OUT_DIR)
elif EXPERIMENT == 2:
    from mech_spoof.experiments.exp2_conflict import run_experiment_2
    result = run_experiment_2(MODEL_KEY, OUT_DIR, exp1_dir=DRIVE_ROOT / slug / 'exp1_authority')
elif EXPERIMENT == 3:
    from mech_spoof.experiments.exp3_refusal import run_experiment_3
    result = run_experiment_3(MODEL_KEY, OUT_DIR, exp1_dir=DRIVE_ROOT / slug / 'exp1_authority')
elif EXPERIMENT == 4:
    from mech_spoof.experiments.exp4_attacks import run_experiment_4
    result = run_experiment_4(
        MODEL_KEY, OUT_DIR,
        exp1_dir=DRIVE_ROOT / slug / 'exp1_authority',
        exp3_dir=DRIVE_ROOT / slug / 'exp3_refusal',
    )
print('done:', OUT_DIR)

In [ ]:
# Inspect + plot (Exp 1 only — per-model plot)
if EXPERIMENT == 1:
    from mech_spoof.viz import plot_layer_accuracy
    plot_layer_accuracy(result.accuracies, MODEL_KEY, OUT_DIR / 'layer_accuracy.png')
    print({l: round(a, 3) for l, a in result.accuracies.items()})

In [ ]:
# Matched-baseline CONTROL for Exp 1.
# Both conditions share the same baseline system block; only the placement of the
# instruction-of-interest varies. Drops the trivial "is there a system block?" shortcut
# that the plain S vs U comparison leaks at layer 0.
if EXPERIMENT == 1:
    from mech_spoof.experiments.exp1_authority import run_experiment_1_control
    import matplotlib.pyplot as plt

    CONTROL_OUT = DRIVE_ROOT / slug / 'exp1_authority_control_matched'
    CONTROL_OUT.mkdir(parents=True, exist_ok=True)
    control_result = run_experiment_1_control(MODEL_KEY, CONTROL_OUT)

    orig_accs = [result.accuracies[i] for i in range(result.n_layers)]
    ctrl_accs = [control_result.accuracies[i] for i in range(control_result.n_layers)]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(orig_accs, marker='o', label='Original (S vs U)')
    ax.plot(ctrl_accs, marker='s', label='Control (matched baseline)')
    ax.axhline(0.5, color='gray', ls='--', lw=0.8, label='chance')
    ax.set_xlabel('layer'); ax.set_ylabel('probe accuracy'); ax.set_ylim(0.4, 1.05)
    ax.set_title(f'{MODEL_KEY}: original vs matched-baseline control')
    ax.legend(); plt.tight_layout()
    plt.savefig(CONTROL_OUT / 'original_vs_control.png', dpi=120)
    plt.show()

    print('CONTROL bundle:', CONTROL_OUT)
    print('original  :', {l: round(a, 3) for l, a in result.accuracies.items()})
    print('control   :', {l: round(a, 3) for l, a in control_result.accuracies.items()})